In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR = Path.home() / "kkbox-churn" / "data" / "parquet"

master = pd.read_parquet(DATA_DIR / "master_dataset.parquet")

drop_cols = [
    'msno', 'last_transaction_date', 'last_expire_date',
    'last_listen_date', 'first_listen_date'
]
master = master.drop(columns=drop_cols)

for col in ['city', 'registered_via', 'bd_bucket', 'auto_renew_switch']:
    master[col] = master[col].astype('category')

# Temporal split — April train, May val
master_raw = pd.read_parquet(DATA_DIR / "master_dataset.parquet")
apr_idx = master_raw[master_raw['last_expire_date'].dt.to_period('M') == '2017-04'].index
may_idx = master_raw[master_raw['last_expire_date'].dt.to_period('M') == '2017-05'].index

X = master.drop(columns=['is_churn'])
y = master['is_churn']

X_train, y_train = X.loc[apr_idx], y.loc[apr_idx]
X_val,   y_val   = X.loc[may_idx], y.loc[may_idx]

print(f"Train: {X_train.shape}, churn rate: {y_train.mean():.4f}")
print(f"Val:   {X_val.shape},   churn rate: {y_val.mean():.4f}")

/opt/anaconda3/envs/kkbox/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: (800236, 101), churn rate: 0.0165
Val:   (79942, 101),   churn rate: 0.0502


In [4]:
# Cell 2 — Optuna tuning
def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'n_estimators': 2000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': 1,
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
        'is_unbalance': True,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=-1)
        ]
    )

    preds = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nBest AUC: {study.best_value:.4f}")
print(f"Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

Best trial: 38. Best value: 0.948066: 100%|██████████| 50/50 [07:47<00:00,  9.36s/it]


Best AUC: 0.9481
Best params:
  learning_rate: 0.05386267542952244
  num_leaves: 39
  min_child_samples: 159
  feature_fraction: 0.8526646900411762
  bagging_fraction: 0.8828494997505784
  reg_alpha: 0.10229096094071723
  reg_lambda: 1.1827367316767054e-05
  min_split_gain: 0.7253877757449226
